# TsModels — 时间序列模型估计工具包

本 Notebook 演示 `TsModels` 包的主要公开工作流，包括：

1. **SARIMAX 估计** — 建模、诊断、预测
2. **GARCH 估计** — 波动率建模（含纯 ARCH / GARCH / GARCH-M）
3. **IGARCH 持久性检验** — 波动率持续性检验
4. **GARCH-M (ARCH-in-Mean)** — 条件波动率进入均值方程
5. **参数估计结果对比** — `compare_models()` Markdown 表格输出
6. **预测功能** — SARIMAX 预测 vs GARCH 波动率预测
7. **与 TsPlots / TsTests 的无缝衔接**
8. **GJR-GARCH** — 非对称 GARCH，杠杆效应
9. **EGARCH** — 指数 GARCH，对数方差建模
10. **STL 分解** — 趋势、季节项、残差与异常值稳健权重
11. **样本外评估** — 单模型 OOS、统一窗口多模型比较与滚动回测

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from Ts.TsSims import simulate_sarima, simulate_garch
from Ts.TsModels import EventSpec, GARCH, SARIMAX
from Ts.TsUtils import STL

---
## 1. SARIMAX 模型估计

### 1.1 AR(1) 估计

In [ ]:
# 生成 AR(1) 数据: phi = 0.7
sim = simulate_sarima(n=200, order=(1, 0, 0), ar=0.7, seed=42, burn=100)
data = sim.data

# 估计 AR(1)
model = SARIMAX(data, order=(1, 0, 0))
result = model.fit()
print(result.summary())
print(f"\n真实 phi = 0.7, 估计 phi_hat = {result.params.get('ar.L1', 'N/A')}")

### 1.2 拟合图

In [ ]:
result.plot_fit()
plt.show()

### 1.3 诊断图 — 残差检验 (含白噪音 + 正态性)

In [ ]:
result.plot_diagnostics()
plt.show()
print("注意: 残差面板右上角标注了白噪音和正态性检验结果")

### 1.4 残差统计检验 — 四项检验 (白噪音 + 正态性 + ARCH-Q + ARCH-LM)

In [ ]:
tests = result.test_residuals(lags=10)
print(tests)

# 单独访问各项检验结果
print("\n白噪音 (raw Ljung-Box):")
print(f"  Q = {tests.white_noise.statistic:.3f}, p = {tests.white_noise.pvalue:.4f}")
print("\n正态性 (Jarque-Bera):")
print(f"  JB = {tests.normality.statistic:.3f}, p = {tests.normality.pvalue:.4f}")
print(f"  Skewness = {tests.normality.skewness:.4f}, Kurtosis = {tests.normality.kurtosis:.4f}")

### 1.5 MA(1) 估计

In [ ]:
sim = simulate_sarima(n=200, order=(0, 0, 1), ma=0.5, seed=456, burn=100)
model = SARIMAX(sim.data, order=(0, 0, 1))
result = model.fit()
print(result.summary())

### 1.6 ARIMA(1,1,0) with trend="ct"

In [ ]:
sim = simulate_sarima(n=200, order=(1, 1, 0), ar=0.3, const=0.1, seed=7)
model = SARIMAX(sim.data, order=(1, 1, 0))
result = model.fit()
print(result.summary())

### 1.7 季节性 SARIMAX

In [ ]:
sim = simulate_sarima(
    n=300, order=(1, 0, 0), ar=0.5,
    seasonal_order=(1, 0, 0, 4), seasonal_ar=0.3,
    seed=42,
)
model = SARIMAX(sim.data, order=(1, 0, 0), seasonal_order=(1, 0, 0, 4))
result = model.fit()
print(result.summary())

### 1.8 SARIMAX 预测

In [ ]:
# SARIMAX 预测 (样本外 20 步)
pr = result.predict(start=len(sim.data), end=len(sim.data) + 19)
print(f"预测均值: {pr.mean[:5]}...")
print(f"95% CI 下界: {pr.lower[:5]}...")
print(f"95% CI 上界: {pr.upper[:5]}...")

# 可视化
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sim.data, linewidth=1, label="Data")
forecast_idx = range(len(sim.data), len(sim.data) + 20)
ax.plot(forecast_idx, pr.mean, color="#D55E00", linewidth=2, label="Forecast")
ax.fill_between(forecast_idx, pr.lower, pr.upper, alpha=0.2, color="#D55E00")
ax.legend()
ax.set_title("SARIMAX 20-Step Forecast with 95% CI")
plt.show()

### 1.9 AR/MA 单位根诊断 — `plot_roots()`

`plot_roots()` 在复平面单位圆上绘制**逆** AR 根和逆 MA 根：

- **蓝色圆点 (●)** = 逆 AR 根 → 全部在单位圆内 = 平稳
- **橙色三角 (▲)** = 逆 MA 根 → 全部在单位圆内 = 可逆

**方法签名：** `plot_roots(title=None) → (fig, ax)`

| 参数 | 类型 | 说明 |
|------|------|------|
| `title` | `str`, optional | 自定义标题。`None` 时自动生成含模型阶数的标题 |
| **返回** | `(Figure, Axes)` | matplotlib 图对象，可进一步修改后保存或显示 |

**相关属性：** `arroots` / `maroots` — 原始 AR/MA 特征根（非逆根）。`1/arroots` 即为图中绘制的逆根。

In [ ]:
# === 基本用法：默认标题 ===
# 先看原始特征根（非逆根）
print(f"AR roots: {result.arroots}")
print(f"AR inverse roots (plotted): {1.0 / result.arroots}")

# === AR(1) 只有 AR 根 ===
ar1_data = simulate_sarima(n=200, order=(1, 0, 0), ar=0.7, seed=42, burn=100).data
ar1_model = SARIMAX(ar1_data, order=(1, 0, 0))
ar1_result = ar1_model.fit()
fig, ax = ar1_result.plot_roots()  # 默认标题: "SARIMAX(1, 0, 0): Inverse AR and MA Roots"
plt.show()

# === 自定义标题 ===
fig, ax = ar1_result.plot_roots(title="AR(1): 平稳性诊断 — 单位根检验")
plt.show()

# === MA(1) 只有 MA 根 ===
ma1_data = simulate_sarima(n=200, order=(0, 0, 1), ma=0.5, seed=456, burn=100).data
ma1_model = SARIMAX(ma1_data, order=(0, 0, 1))
ma1_result = ma1_model.fit()
print(f"MA roots: {ma1_result.maroots}")
fig, ax = ma1_result.plot_roots(title="MA(1): 可逆性诊断")
plt.show()

# === ARMA(1,1): 同时显示 AR 和 MA 逆根 ===
arma_data = simulate_sarima(n=200, order=(1, 0, 1), ar=[0.7], ma=[0.5], seed=42, burn=100).data
arma_model = SARIMAX(arma_data, order=(1, 0, 1))
arma_result = arma_model.fit()
fig, ax = arma_result.plot_roots()

# 利用返回值进一步自定义图形
ax.set_title("ARMA(1,1): AR + MA 逆根诊断", fontsize=16, color="#333333")
ax.annotate("单位圆内 = 平稳 + 可逆", xy=(0.05, 0.95), xycoords="axes fraction",
            fontsize=11, color="#666666", va="top")
plt.show()
# AR 根 (蓝色圆点) 和 MA 根 (橙色三角) 均落在单位圆内 → 平稳且可逆

### 1.10 稀疏滞后、周期与长期均衡

`order` 中的 AR/MA 阶数既可使用连续整数，也可显式给出参与估计的滞后。
`cycle_period()` 是 AR(2) 复根条件下的代数周期诊断；`long_run_equilibrium()`
只在无差分、无时间趋势且 AR 多项式平稳时有定义。

In [ ]:
# 稀疏 AR：只估计 L1 和 L3，严格固定 L2 = 0
sparse_data = simulate_sarima(
    n=300,
    order=(3, 0, 0),
    ar=[0.60, 0.0, -0.25],
    seed=2026,
    burn=150,
).data
sparse_result = SARIMAX(sparse_data, order=([1, 3], 0, 0)).fit()

# 具有复共轭根的 AR(2) 阻尼周期
cycle_data = simulate_sarima(
    n=400,
    order=(2, 0, 0),
    ar=[1.0, -0.5],
    seed=27,
    burn=200,
).data
cycle_result = SARIMAX(cycle_data, order=(2, 0, 0)).fit()
cycle = cycle_result.cycle_period()

print("Active AR lags:", sparse_result.ar_lags)
print("Fixed coefficients:", sparse_result.fixed_params)
print("Cycle identified:", cycle.identified)
print("Estimated cycle period:", f"{cycle.period:.3f}" if cycle.period else "N/A")
print("AR(1) long-run equilibrium:", ar1_result.long_run_equilibrium())

assert sparse_result.ar_lags == (1, 3)
assert "ar.L2" in sparse_result.fixed_params
assert cycle.identified

### 1.11 日期、普通外生变量与事件设计

`SARIMAX` 对 pandas 输入按日期严格对齐。DataFrame `exog` 可以同时包含历史行和
样本后的未来行；历史部分进入估计，未来部分成为默认预测路径。事件变量由
`EventSpec` 生成，不与普通外生变量混在调用方的数据表中。

In [ ]:
rng = np.random.default_rng(42)
sarimax_dates = pd.date_range("2020-01-01", periods=60, freq="MS")
sarimax_all_dates = pd.date_range(sarimax_dates[0], periods=66, freq="MS")
sarimax_future_dates = sarimax_all_dates[-6:]

sarimax_controls = pd.DataFrame(
    {
        "rate": np.sin(np.arange(66) / 7.0),
        "income": np.linspace(-1.0, 1.0, 66),
    },
    index=sarimax_all_dates,
)
policy_position = 30
policy_level = (np.arange(60) >= policy_position).astype(float)
implementation_effect = np.zeros(60)
implementation_effect[policy_position + 1] = 0.4
sarimax_y = pd.Series(
    2.0
    + 0.8 * sarimax_controls.loc[sarimax_dates, "rate"].to_numpy()
    - 0.5 * sarimax_controls.loc[sarimax_dates, "income"].to_numpy()
    + 1.2 * policy_level
    + implementation_effect
    + rng.normal(scale=0.12, size=60),
    index=sarimax_dates,
    name="outcome",
)

sarimax_events = (
    EventSpec("announcement", [sarimax_dates[24]], "pulse", date_rule="exact"),
    EventSpec("policy", [sarimax_dates[policy_position]], "step", date_rule="exact"),
    EventSpec(
        "implementation",
        [sarimax_dates[policy_position]],
        "pulse",
        window=(-2, 2),
        reference=-1,
        date_rule="exact",
    ),
)
sarimax_model = SARIMAX(
    sarimax_y,
    exog=sarimax_controls,
    events=sarimax_events,
    order=(0, 0, 0),
    trend="c",
)
sarimax_result = sarimax_model.fit()

print("普通外生变量:", sarimax_result.exog_names)
print("事件:", sarimax_result.event_names)
print("合并设计矩阵:", sarimax_result.design_columns)
print("rate 估计系数:", f"{sarimax_result.params['rate']:.3f}")

assert sarimax_model.future_exog.index.equals(sarimax_future_dates)
assert sarimax_result.exog_names == ("rate", "income")
assert set(sarimax_result.event_names) == {"announcement", "policy", "implementation"}

### 1.12 默认未来路径与命名情景

不传 `future_exog` 时使用构造模型时保存的未来 DataFrame。传入映射时返回
`ScenarioForecastResult`；因为模型已保存默认路径，结果同时包含 `default`、
`baseline` 和 `stress`。

In [ ]:
default_forecast = sarimax_result.predict(start=60, end=65)

baseline = sarimax_controls.loc[sarimax_future_dates].copy()
stress = baseline.copy()
stress["rate"] += 0.75
scenario_forecast = sarimax_result.predict(
    start=60,
    end=65,
    future_exog={"baseline": baseline, "stress": stress},
)

print(scenario_forecast.summary())
print("默认路径首期预测:", f"{default_forecast.mean[0]:.3f}")
print(
    "压力情景减基准情景:",
    np.round(
        scenario_forecast["stress"].mean - scenario_forecast["baseline"].mean,
        3,
    ),
)
scenario_forecast.plot(title="SARIMAX 未来外生变量情景")
plt.show()

assert tuple(scenario_forecast.scenarios) == ("default", "baseline", "stress")
assert np.all(scenario_forecast["stress"].mean > scenario_forecast["baseline"].mean)

### 1.13 政策效果与输入契约

`policy_effect()` 比较保留普通控制变量和其他事件不变时的事实/反事实路径。
这是给定模型设定下的条件效果；只有额外的外生性、无遗漏同期冲击等识别条件成立时，
才能解释为因果政策效果。缺失或错位的历史外生日期会直接报错。

In [ ]:
policy_effect = sarimax_result.policy_effect("policy", method="delta")
print(policy_effect.summary())
print("识别说明:", policy_effect.identification_note)

misaligned_controls = sarimax_controls.drop(index=sarimax_dates[5])
try:
    SARIMAX(sarimax_y, exog=misaligned_controls)
except ValueError as error:
    print("预期的日期对齐错误:", error)
else:
    raise AssertionError("错位的历史外生变量必须被拒绝")

assert policy_effect.method == "delta"
assert np.isfinite(policy_effect.cumulative_effect)

---
## 2. 纯 ARCH 估计 (GARCH with q=0)

### 2.1 ARCH(1) 估计 — GARCH(p=1, q=0)

In [ ]:
# 生成 ARCH(1): omega = 0.4, alpha1 = 0.5
sim = simulate_garch(n=400, p=1, q=0, omega=0.4, alpha=0.5, seed=789, burn=300)

model = GARCH(sim.data, p=1, q=0)
result = model.fit()
print(result.summary())
print("\n真实 omega=0.4, alpha1=0.5")

### 2.2 条件波动率可视化

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5))
ax1.plot(sim.data, linewidth=0.8)
ax1.set_title("ARCH(1) Simulated Returns")
ax2.plot(result.conditional_volatility, color="#D55E00", linewidth=1.5)
ax2.set_title("Estimated Conditional Volatility (sigma_t)")
plt.tight_layout()
plt.show()

### 2.3 ARCH with AR mean — GARCH(p=1, q=0, mean="AR")

In [ ]:
sim = simulate_garch(
    n=400, p=1, q=0, omega=0.4, alpha=0.5,
    mean_model="ar", mean_ar=0.6, mean_const=0.5,
    seed=42, burn=200,
)
model = GARCH(sim.data, p=1, q=0, mean="AR")
result = model.fit()
print(result.summary())

### 2.4 ARCH(2) 估计 — GARCH(p=2, q=0)

In [ ]:
sim = simulate_garch(n=400, p=2, q=0, omega=0.2, alpha=[0.3, 0.2], seed=10, burn=200)
model = GARCH(sim.data, p=2, q=0)
result = model.fit()
print(result.summary())

---
## 3. GARCH 模型估计

### 3.1 GARCH(1,1) 估计 + IGARCH 持久性检验

In [ ]:
# GARCH(1,1): omega=0.1, alpha=0.2, beta=0.7 (持久性=0.9, 平稳)
sim = simulate_garch(
    n=500, p=1, q=1, omega=0.1, alpha=0.2, beta=0.7,
    seed=123, burn=300,
)
model = GARCH(sim.data, p=1, q=1)
result = model.fit()
print(result.summary())  # summary() 自动包含 IGARCH 持久性检验 + 分布信息

### 3.2 test_persistence() 详细输出

In [ ]:
stab_test = result.test_persistence()
for k, v in stab_test.items():
    print(f"{k:<20s} = {v}")

### 3.3 IGARCH 边界检验 — 接近 1 时的行为

In [ ]:
# 构造 alpha+beta 接近 1 的 GARCH
sim_igarch = simulate_garch(
    n=500, p=1, q=1, omega=0.05, alpha=0.15, beta=0.83,
    seed=456, burn=300,
)
model2 = GARCH(sim_igarch.data, p=1, q=1)
result2 = model2.fit()
print(result2.summary())
# 注意 IGARCH 检验结论 — alpha+beta=0.98 时可能无法拒绝 H0

### 3.4 Student's t 分布 GARCH — 厚尾新息

`GARCH(data, dist="t")` 即可使用 Student's t 分布。
`summary()` 标题行自动标注分布类型 `[Student's t]`。

In [ ]:
sim_t = simulate_garch(
    n=500, p=1, q=1, omega=0.1, alpha=0.2, beta=0.7,
    dist="t", dist_params={"df": 5},
    seed=42, burn=300,
)
model_t = GARCH(sim_t.data, p=1, q=1, dist="t")
result_t = model_t.fit()
print(result_t.summary())
# 注意 nu 参数 — 估计的 t 分布自由度

### 3.5 GARCH 分布类型对比

`dist` 参数支持: `"normal"`, `"t"`, `"skewt"`, `"ged"`。

In [ ]:
distributions = ["normal", "t", "ged"]
for dist in distributions:
    model = GARCH(sim.data, p=1, q=1, dist=dist)
    r = model.fit()
    print(f"dist={dist:<8s}  AIC={r.aic:.2f}  BIC={r.bic:.2f}")

### 3.6 IGARCH 约束估计 — `igarch=True`

IGARCH (Integrated GARCH) 强制约束 sum(alpha) + sum(beta) = 1。
在此约束下，波动率冲击具有永久性，预测方差随 horizon 线性增长。

In [ ]:
# IGARCH(1,1) 约束估计 — alpha+beta=1 硬约束
from Ts.TsSims import simulate_igarch

# 生成 IGARCH(1,1) 数据: alpha=0.30, beta=0.70 (自动满足约束)
sim_ig = simulate_igarch(n=500, p=1, q=1, omega=0.05, alpha=[0.30], seed=42)
print(f"simulate_igarch 约束验证: sum(alpha)+sum(beta) = {sum(sim_ig.params['alpha']) + sum(sim_ig.params['beta']):.10f}")

# 用 igarch=True 估计 IGARCH 模型
model_ig = GARCH(sim_ig.data, p=1, q=1, igarch=True)
result_ig = model_ig.fit()
print(result_ig.summary())

# 验证估计结果也满足约束
alpha_hat = sum(v for k, v in result_ig.params.items() if k.startswith("alpha"))
beta_hat = sum(v for k, v in result_ig.params.items() if k.startswith("beta"))
print(f"\n[验证] 估计约束: alpha+beta = {alpha_hat + beta_hat:.10f}")

### 3.6.1 IGARCH 预测：方差线性增长

与标准 GARCH 不同，IGARCH 的波动率预测随 horizon 线性增长而不回归均值。

In [ ]:
# IGARCH vs GARCH 预测对比
# IGARCH 预测: 方差线性增长，无均值回归
pr_ig = result_ig.predict(start=result_ig.nobs, end=result_ig.nobs + 19)

# 对比: 标准 GARCH 预测 (均值回归)
model_g = GARCH(sim_ig.data, p=1, q=1, igarch=False)
result_g = model_g.fit()
pr_g = result_g.predict(start=result_g.nobs, end=result_g.nobs + 19)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(pr_ig.mean, color="#D55E00", linewidth=2, marker="o", markersize=3, label="IGARCH (linear growth)")
ax.plot(pr_g.mean, color="#0072B2", linewidth=2, marker="s", markersize=3, label="GARCH (mean-reverting)")
ax.set_title("IGARCH vs GARCH: 20-Step Volatility Forecast")
ax.set_ylabel("sigma_{T+h}")
ax.legend()
plt.show()
# IGARCH: 方差随 h 线性增长 (omega * h + sigma2_T)
# GARCH: 方差回归到无条件均值

---
## 4. 波动率预测

In [ ]:
# ARCH 波动率预测
pr = result.predict(start=result.nobs, end=result.nobs + 49)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(pr.mean, color="#D55E00", linewidth=2, marker="o", markersize=3)
ax.axhline(pr.mean[-1], color="gray", linestyle=":", label=f"Final sigma = {pr.mean[-1]:.3f}")
ax.set_title("GARCH(1,1) 50-Step Volatility Forecast")
ax.set_ylabel("sigma_{T+h}")
ax.legend()
plt.show()

---
## 5. 工作流集成

TsSims -> TsModels -> TsPlots / TsTests 的完整链条。
诊断图自动显示白噪音和正态性检验结果。

In [ ]:
# Step 1: 模拟数据
sim = simulate_garch(n=500, p=1, q=1, omega=0.1, alpha=0.2, beta=0.7, seed=42)

# Step 2: 估计模型
model = GARCH(sim.data, p=1, q=1)
result = model.fit()

# Step 3a: 绘图诊断 (残差面板自动显示白噪音 + 正态性检验)
result.plot_diagnostics()
plt.suptitle("GARCH(1,1) Diagnostic Plots", y=1.02)
plt.show()

# Step 3b: 四项统计检验
tests = result.test_residuals(lags=10)
print(tests)

# Step 3c: IGARCH 持久性检验
stab = result.test_persistence()
print("\n波动率持久性:")
print(f"  persistence_sum = {stab['persistence_sum']:.4f}")
print(f"  chi2(1) = {stab['chi2']:.3f}, p = {stab['pvalue']:.4f}")
print(f"  结论: {stab['conclusion']}")

| `SARIMAXResult` | `summary()`, `plot_fit()`, `plot_diagnostics()`, `test_residuals()` | `predict()`, `arroots`, `maroots`, `plot_roots()` |
| `GARCHResult` | 同上 + `dist`, `garch_m`, `garch_m_form` | `predict()`, `test_persistence()`, `conditional_volatility` |
| `VARResult` | 同上（2-D fitted_values/residuals） | `irf()` → `IRFResult`, `oirf()`, `fevd()` → `FEVDResult`, `plot_irf()`, `granger_causality()`, `predict()`, `is_stable`, `plot_roots()` |
| `SVARResult` | 同上（继承 VARResult）+ `A`, `B`, `sigma_u` | `sirf()` → 结构性 IRF, `structural_residuals` |
| `VECMResult` | `summary()`（Stata风格）, `plot_diagnostics()`, `test_residuals()` | `irf()`, `fevd()`, `predict()`, `granger_causality()`, `is_stable`, `plot_roots()`, `alpha`/`beta`/`gamma`/`sigma_u` |

---

## 6. GARCH-M (ARCH-in-Mean) 估计

GARCH-M 将条件波动率引入均值方程：

$$
y_t = \mu + \lambda \cdot \sigma_t + \varepsilon_t
$$

ARCH-in-mean 系数在结果中对应参数 `kappa`。

In [ ]:
# GARCH-M(1,1): sigma_t 进入均值方程
sim = simulate_garch(
    n=500, p=1, q=1, omega=0.1, alpha=0.2, beta=0.7,
    seed=42, burn=300,
)
model_m = GARCH(sim.data, p=1, q=1, garch_m=True)
result_m = model_m.fit()
print(result_m.summary())
# 注意 kappa 参数 — 均值方程中条件波动率的系数

### 6.1 GARCH-M 形式对比

`garch_m_form` 控制 sigma 进入均值方程的形式：

| 值 | 均值方程项 |
|----|-----------|
| `"vol"` | $\lambda \cdot \sigma_t$ |
| `"var"` | $\lambda \cdot \sigma_t^2$ |
| `"log"` | $\lambda \cdot \log(\sigma_t^2)$ |

In [ ]:
forms = ["vol", "var", "log"]
for form in forms:
    model = GARCH(sim.data, p=1, q=1, garch_m=True, garch_m_form=form)
    r = model.fit()
    kappa = r.params.get("kappa", np.nan)
    print(f"form={form:<5s}  kappa={kappa:<12.6f}  AIC={r.aic:.2f}  BIC={r.bic:.2f}")

---

## 7. 多模型结果对比 — `compare_models`

支持将多个模型的 AIC、BIC、Log-Likelihood、参数估计值及显著性星号
整合为 Markdown 表格，便于横向比较。

In [ ]:
from Ts.TsModels import compare_models

# 拟合多个模型
model_base = GARCH(sim.data, p=1, q=1)
result_base = model_base.fit()

model_m = GARCH(sim.data, p=1, q=1, garch_m=True)
result_m = model_m.fit()

model_t = GARCH(sim.data, p=1, q=1, dist="t")
result_t = model_t.fit()

# Markdown 对比表格
table = compare_models({
    "GARCH(1,1) [Normal]": result_base,
    "GARCH-M(1,1) [Normal]": result_m,
    "GARCH(1,1) [t]": result_t,
})
print(table)
# 参数估计值含标准误和显著性星号 (*** p<0.01, ** p<0.05, * p<0.1)

---

## 8. GJR-GARCH（非对称 GARCH）估计

GJR-GARCH (Glosten-Jagannathan-Runkle) 通过 `o` 参数引入杠杆效应。
`o>=1` 时，负向冲击对波动率的影响大于正向冲击（gamma 系数）。

In [ ]:
# GJR-GARCH(1,1,1) 估计
np.random.seed(123)
n = 500
omega, alpha, gamma, beta = 0.05, 0.10, 0.15, 0.75
eps = np.random.randn(n)
sigma2 = np.zeros(n)
e = np.zeros(n)
sigma2[0] = omega / (1 - alpha - 0.5 * gamma - beta)
e[0] = np.sqrt(sigma2[0]) * eps[0]
for t in range(1, n):
    I_neg = 1.0 if e[t - 1] < 0 else 0.0
    sigma2[t] = omega + alpha * e[t - 1]**2 \
               + gamma * I_neg * e[t - 1]**2 \
               + beta * sigma2[t - 1]
    e[t] = np.sqrt(max(sigma2[t], 1e-10)) * eps[t]
y = 1.0 + e

model_gjr = GARCH(y, p=1, o=1, q=1)
result_gjr = model_gjr.fit()
print(result_gjr.summary())
# gamma[1] > 0 表示杠杆效应: 负向冲击增加波动率

---

## 9. EGARCH（指数 GARCH）估计

EGARCH (Nelson 1991) 对对数方差建模，天然保证方差为正。
通过 `vol="EGARCH"` 参数启用，`o>=1` 引入非对称杠杆效应。

In [ ]:
# EGARCH(1,1,1) 估计
np.random.seed(123)
n = 500
omega, alpha, gamma, beta = 0.05, 0.20, 0.10, 0.30
eps = np.random.randn(n)
ln_sigma2_sim = np.zeros(n)
e_sim = np.zeros(n)
ln_sigma2_sim[0] = omega
sigma2_0 = np.exp(ln_sigma2_sim[0])
e_sim[0] = np.sqrt(sigma2_0) * eps[0]
for t in range(1, n):
    z = eps[t]
    ln_sigma2_sim[t] = (omega
                       + alpha * (abs(z) - np.sqrt(2.0 / np.pi))
                       + gamma * z
                       + beta * ln_sigma2_sim[t - 1])
    e_sim[t] = np.sqrt(np.exp(ln_sigma2_sim[t])) * z
y_egarch = 1.0 + e_sim

model_egarch = GARCH(y_egarch, p=1, o=1, q=1, vol="EGARCH")
result_egarch = model_egarch.fit()
print(result_egarch.summary())
# EGARCH 持久性仅检验 sum(beta)=1，不含 alpha/gamma

---

## 10. AutoSARIMAX / AutoGARCH — 自动最优参数选择

`AutoSARIMAX` 和 `AutoGARCH` 通过网格搜索自动选择使信息准则 (AIC/BIC/HQIC/AICC)
最小的模型阶数。`AutoGARCH` 支持全部 GARCH 族变体：标准 GARCH、GJR-GARCH、
EGARCH、IGARCH、GARCH-M。

### 10.1 AutoSARIMAX — 自动选择最优 SARIMAX 阶数

In [ ]:
from Ts.TsModels import AutoGARCH, AutoSARIMAX
from Ts.TsSims import simulate_egarch, simulate_garch_m, simulate_igarch

# 无普通外生变量时，SARIMAX 阶数搜索覆盖 SARIMA 特例
sim_ar = simulate_sarima(n=200, order=(1, 0, 0), ar=[0.7], seed=42)
auto_ar = AutoSARIMAX(
    sim_ar.data,
    p=(0, 2),
    d=(0, 1),
    q=(0, 2),
    P=(0, 0),
    D=(0, 0),
    Q=(0, 0),
    criterion="aic",
)
result_ar = auto_ar.fit()

# 外生变量、事件和默认未来路径传给每一个候选模型
auto_x = AutoSARIMAX(
    sarimax_y,
    exog=sarimax_controls,
    events=sarimax_events,
    p=(0, 1),
    d=(0, 0),
    q=(0, 0),
    P=(0, 0),
    D=(0, 0),
    Q=(0, 0),
    criterion="bic",
)
result_x = auto_x.fit()

print(result_ar.summary())
print("外生模型最优阶数:", result_x.best_order)
print("外生变量名称:", result_x.best_result.exog_names)
assert result_x.best_result.exog_names == ("rate", "income")

### 10.2 AutoGARCH — EGARCH 自动选择

`vol="EGARCH"` 启用 EGARCH 模型的自动阶数选择。

In [ ]:
# AutoGARCH + EGARCH: 自动搜索最优 EGARCH(p,o,q) 阶数
sim_eg = simulate_egarch(n=300, p=1, q=1, o=1, seed=42, burn=200)
auto_eg = AutoGARCH(sim_eg.data, p=(1, 2), q=(1, 2), o=(1, 1),
                    vol="EGARCH", criterion="aic")
result_eg = auto_eg.fit()
print(result_eg.summary())

### 10.3 AutoGARCH — IGARCH 自动选择

`igarch=True` 启用 IGARCH 约束估计的自动阶数选择。

In [ ]:
# AutoGARCH + IGARCH: 自动搜索最优 IGARCH(p,q) 阶数
sim_ig = simulate_igarch(n=300, p=1, q=1, omega=0.10, alpha=[0.20], seed=42, burn=200)
auto_ig = AutoGARCH(sim_ig.data, p=(1, 2), q=(1, 2),
                    igarch=True, criterion="bic")
result_ig = auto_ig.fit()
print(result_ig.summary())

### 10.4 AutoGARCH — GARCH-M 自动选择

`garch_m=True` 启用 GARCH-M (ARCH-in-mean) 的自动阶数选择。

In [ ]:
# AutoGARCH + GARCH-M: 自动搜索最优 GARCH-M(p,q) 阶数
sim_m = simulate_garch_m(n=300, p=1, q=1, omega=0.10, alpha=[0.20], beta=[0.60],
                         garch_m_kappa=0.20, seed=42, burn=200)
auto_m = AutoGARCH(sim_m.data, p=(1, 2), q=(1, 2),
                   garch_m=True, criterion="aic")
result_m = auto_m.fit()
print(result_m.summary())

---

## 11. VAR — 向量自回归

VAR 是多变量时间序列分析的基础模型。每个变量由其自身滞后值和其他变量的滞后值共同解释。

In [ ]:
import numpy as np
from Ts.TsModels import VAR
from Ts.TsSims import simulate_sarima

# 生成两个相关的 AR(1) 过程作为 VAR 数据
r0 = simulate_sarima(n=200, order=(1, 0, 0), ar=[0.7], seed=42, burn=100)
r1 = simulate_sarima(n=200, order=(1, 0, 0), ar=[0.5], seed=99, burn=100)
data_2d = np.column_stack([r0.data, r1.data])

# VAR(2) 估计
model = VAR(data_2d, lags=2, cols=["y0", "y1"])
result = model.fit()
print(result.summary())

### 11.1 滞后阶数选择

`VAR.select_order()` 静态方法基于信息准则选择最优滞后阶数。

In [ ]:
order_info = VAR.select_order(data_2d, max_lags=8, criterion="aic")
print(order_info)

### 11.2 脉冲响应函数 (IRF)

IRF 展示一个变量的冲击如何随时间影响其他变量。

In [ ]:
# === 非正交化 IRF — 返回 IRFResult（含 .values / .lower / .upper / .summary() / .get()） ===
irf_result = result.irf(periods=10, orth=False)
print(f"Raw IRF values shape: {irf_result.values.shape}  (periods+1, k, k)")
# irf_result.values[h, i, j] = 变量 i 对冲击 j 在第 h 期的响应

# === 正交化 IRF (Cholesky 分解) — OIRF ===
irf_orth_result = result.irf(periods=10, orth=True)
print(f"Orth IRF values shape: {irf_orth_result.values.shape}  (periods+1, k, k)")

# === 正交化 IRF 的显式调用方式 ===
oirf_result = result.irf(periods=10, orth=True)

# === 提取单个响应路径: y0 对 y1 冲击的响应 ===
d = irf_result.get("y0", "y1")
print(f"\ny0 -> y1 shock: step={d['step'][:4]}, value={d['value'][:4]}")
print(f"  95% CI 下界: {d['lower'][:4]}")
print(f"  95% CI 上界: {d['upper'][:4]}")

# === 查看 Stata 风格表格 ===
print("\n" + irf_result.summary())

# 验证: 正交 IRF 与原始 IRF 不同
print(f"\nOIRF differs from raw: {not np.allclose(irf_result.values, irf_orth_result.values)}")

# 第二次调用 (相同 periods) 自动从缓存读取，不重复计算
irf_result2 = result.irf(periods=10, orth=False)
print(f"Cached result matches: {np.allclose(irf_result.values, irf_result2.values)}")

# 绘制 IRF (含 95% 置信带)
fig, axes = result.plot_irf(periods=10, orth=False)
plt.show()

# 绘制正交化 IRF
fig, axes = result.plot_irf(periods=10, orth=True)
plt.show()

### 11.3 预测误差方差分解 (FEVD) 与 Granger 因果检验

In [ ]:
# 方差分解 — 返回 FEVDResult（含 Monte Carlo 置信区间 + .summary() + .get()）
fevd_result = result.fevd(periods=10, n_draws=200, seed=42)
print(f"FEVD values shape: {fevd_result.values.shape}  (periods, k, k)")

# === Stata 风格表格输出 ===
print(fevd_result.summary())

# === 提取单个响应-冲击对的方差分解 ===
d = fevd_result.get("y0", "y1")
print("\ny0 <- y1 shock FEVD:")
print(f"  step: {d['step']}")
print(f"  value: {d['value']}")
print(f"  lower: {d['lower']}")
print(f"  upper: {d['upper']}")

# 全部两两 Granger 因果检验 (等价于 Stata 的 vargranger)
print()
print(result.granger_causality(kind="chi2"))

### 11.4 预测与诊断

In [ ]:
# 多步预测 (含 95% 置信区间)
pr = result.predict(start=result.nobs, end=result.nobs + 7)
print(f"预测均值 shape: {pr.mean.shape}  (steps, k)")

# 诊断图: 每个变量的残差 + ACF + PACF
result.plot_diagnostics()
plt.show()

# 拟合图: 实际值 vs 拟合值
result.plot_fit()
plt.show()

# 残差检验: 每变量独立的四项诊断
resid_tests = result.test_residuals(lags=10)
for name, tests in resid_tests.items():
    print(f"\n=== {name} 残差诊断 ===")
    print(tests)

### 11.5 稳定性诊断 — `plot_roots()` 单位根图

`is_stable` 属性检查 VAR 平稳性；`plot_roots()` 在复平面单位圆上可视化
逆特征根。

**方法签名：** `plot_roots(title=None) → (fig, ax)`

| 参数 | 类型 | 说明 |
|------|------|------|
| `title` | `str`, optional | 自定义标题。`None` 时自动生成含稳定性结论的标题 |
| **返回** | `(Figure, Axes)` | matplotlib 图对象，可进一步修改后保存或显示 |

**相关属性：** `is_stable` — bool，所有逆特征根模长 < 1 时返回 `True`。

与 SARIMAX 的 `plot_roots()` 的区别：VAR 只有一种根（特征多项式的逆根），
用**蓝色圆点**绘制，没有 AR/MA 的图形区分。

In [ ]:
# === 基本用法：默认标题（含稳定性结论） ===
print(f"VAR is stable: {result.is_stable}")  # True = 所有逆根在单位圆内

fig, ax = result.plot_roots()  # 默认标题: "VAR(2): Inverse Roots (Stable)"
plt.show()

# === 自定义标题 ===
fig, ax = result.plot_roots(title="VAR(2) 协方差平稳性诊断")
plt.show()

# === 利用返回值自定义图形 ===
fig, ax = result.plot_roots()
ax.set_title("VAR(2): 逆特征根单位圆检验", fontsize=16, color="#333333")
# 标注模长最大的根
roots = result._var_result.roots
max_mod = np.max(np.abs(roots))
ax.annotate(f"max|λ| = {max_mod:.4f}", xy=(0.05, 0.95),
            xycoords="axes fraction", fontsize=11, color="#666666", va="top")
plt.show()
# 所有点落在单位圆内 (|z| < 1) = VAR 协方差平稳

---
## 12. SVAR — 结构向量自回归

SVAR 在简化式 VAR 基础上施加识别约束，恢复结构性冲击。
支持短期约束 (A/B 矩阵) 和长期约束 (Blanchard-Quah)。

In [ ]:
from Ts.TsModels import SVAR
import numpy as np

# === 短期约束 AB-model（Cholesky / 递归识别） ===
A = np.array([[1, 0], [np.nan, 1]])         # 下三角 A，1 个自由参数
B = np.array([[np.nan, 0], [0, np.nan]])     # 对角 B，2 个自由参数

svar_ab = SVAR(data_2d, lags=2, A=A, B=B)
result_ab = svar_ab.fit()
print(result_ab.summary())

# 估计的 A 和 B 矩阵
print(f"\nA matrix:\n{result_ab.A}")
print(f"\nB matrix:\n{result_ab.B}")

# 验证: A^(-1) B B^T A^(-T) = Sigma_u
sigma_implied = np.linalg.inv(result_ab.A) @ result_ab.B @ result_ab.B.T @ np.linalg.inv(result_ab.A).T
print(f"\nA^(-1) B B' A^(-T) equals Sigma_u: {np.allclose(sigma_implied, result_ab.sigma_u, atol=1e-10)}")

# 结构冲击正交性验证
eps = result_ab.structural_residuals
cov_eps = np.cov(eps, rowvar=False)
print(f"\nStructural shock covariance (should be I):\n{cov_eps}")

# === 结构性脉冲响应 (SIRF) ===
sirf = result_ab.irf(periods=10, orth=True, n_draws=50, seed=42)
print(f"\nSIRF shape: {sirf.values.shape}  (periods+1, k, k)")

# SIRF 与简化式 IRF 对比
rirf = result_ab.irf(periods=10, orth=False)
print(f"SIRF differs from reduced-form IRF: {not np.allclose(sirf.values, rirf.values)}")

# 即期冲击矩阵 = A^(-1) B
impact = np.linalg.inv(result_ab.A) @ result_ab.B
print(f"\nImpact matrix (Theta_0 = A^(-1) B):\n{impact}")
print(f"SIRF[0] equals impact: {np.allclose(sirf.values[0], impact, atol=1e-10)}")

# === 长期约束 (Blanchard-Quah) ===
C_lr = np.array([[np.nan, 0], [np.nan, np.nan]])  # C[0,1]=0: 需求冲击对产出的长期效应为零
svar_lr = SVAR(data_2d, lags=2, C_lr=C_lr)
result_lr = svar_lr.fit()
print("\n=== Long-run BQ ===")
print(f"B matrix:\n{result_lr.B}")

# 验证: Psi(1) @ B 是下三角 (长期零约束)
coefs = result_lr._var_result.coefs
A_sum = np.sum(coefs, axis=0)
psi1 = np.linalg.inv(np.eye(2) - A_sum)
lr_impact = psi1 @ result_lr.B
print(f"\nPsi(1) @ B (should be lower triangular):\n{lr_impact}")

# === 继承自 VARResult 的方法 ===
print(f"\nVAR stability: {result_ab.is_stable}")
gc = result_ab.granger_causality(caused=0, causing=1)
print(f"Granger causality p-value: {gc[0].p_value:.4f}")

---
## 13. VECM — 向量误差修正模型

VECM 是协整 VAR 系统的误差修正表示。当 Johansen 检验确认变量间存在
协整关系后，VECM 将 VAR 重新参数化为差分形式 + 误差修正项。

```python
VECM(data, lags=2, coint_rank=1, trend="c", cols=None)
```

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `data` | array-like (nobs, k) | — | 多变量时间序列 |
| `lags` | int | `2` | VAR 水平滞后阶数 (>= 2) |
| `coint_rank` | int | `1` | 协整秩 (1 <= r < k) |
| `trend` | str | `"c"` | `"n"` / `"c"` / `"ct"` |
| `cols` | list of str | `None` | 变量列名 |

VECM 通过 statsmodels VECM 采用 MLE 估计。
`summary()` 输出 Stata 风格格式：模型信息 + 方程汇总表 + 系数表 + beta 协整方程。

In [ ]:
from Ts.TsModels import VECM
import numpy as np

# 生成 3 变量协整数据 (rank=2)
np.random.seed(42)
n = 300
x = np.cumsum(np.random.randn(n))
y = 2.0 * x + np.random.randn(n) * 0.5
z = 3.0 * x + np.random.randn(n) * 0.5
data = np.column_stack([z, y, x])

# VECM 估计: lags=2, coint_rank=2
model = VECM(data, lags=2, coint_rank=2, trend="c", cols=["Z", "Y", "X"])
result = model.fit()
print(result.summary())

# 关键参数矩阵
print(f"\nalpha shape: {result.alpha.shape}  (k x r)")
print(f"beta shape:  {result.beta.shape}  (k x r)")
print(f"gamma shape: {result.gamma.shape}  (k x k*(p-1))")

# 脉冲响应 (正交化)
irf = result.irf(periods=10, orth=True)

# 方差分解
fevd = result.fevd(periods=10)

# Granger 因果检验
gc = result.granger_causality(caused="Z", causing="Y", kind="chi2")
print(f"\nGranger causality (Y -> Z): p={gc.tests[0].p_value:.4f}")

# 稳定性
print(f"VECM is stable: {result.is_stable}")

# 残差诊断
result.test_residuals(lags=10)

---
## 14. 真实样本外评估 (Out-of-Sample Evaluation)

`model.oos(estimation_period=..., validation_period=...)` 会克隆模型，仅使用估计期的数据重新估计，
跨过估计期与验证期之间的间隔预测，并只对验证期计分。返回的 `OOSResult` 包含 `actual`、`mean`、置信区间、
估计期与验证期索引及 MAE、MSE、RMSE、MAPE、sMAPE、Theil U1 等指标。

`predict()` 只负责已拟合模型的拟合值或未来预测，不再承担性能评估。

### 14.1 SARIMAX 真实样本外评估

前 70% 数据用于估计，后 30% 是模型拟合时从未见过的持出区间。

In [ ]:
# 带外生变量和事件的 SARIMAX 真实样本外评估
evaluation_model = SARIMAX(
    sarimax_y,
    exog=sarimax_controls,
    events=sarimax_events,
    order=(0, 0, 0),
    trend="c",
)
evaluation = evaluation_model.oos(
    estimation_period=(sarimax_dates[0], sarimax_dates[44]),
    validation_period=(sarimax_dates[45], sarimax_dates[-1]),
)

print("=== SARIMAX 样本外评估指标 ===")
for name, value in evaluation.metrics.items():
    print(f"  {name:<10s}: {value:.4f}")
print("训练期:", evaluation.estimation_dates[0], "到", evaluation.estimation_dates[-1])
print("验证期:", evaluation.validation_dates[0], "到", evaluation.validation_dates[-1])

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(sarimax_y.index, sarimax_y, color="black", linewidth=0.8,
        alpha=0.55, label="Actual")
ax.plot(evaluation.validation_dates, evaluation.mean, color="#D55E00",
        linewidth=1.5, label="OOS forecast")
if evaluation.lower is not None and evaluation.upper is not None:
    ax.fill_between(evaluation.validation_dates, evaluation.lower, evaluation.upper,
                    alpha=0.15, color="#D55E00")
ax.axvline(evaluation.validation_dates[0], color="gray", linestyle="--",
           linewidth=0.8, label="Validation start")
ax.legend(fontsize=8)
ax.set_title("SARIMAX Leakage-Free OOS Forecast with Exog")
plt.show()

assert evaluation.validation_indices.tolist() == list(range(45, 60))
assert evaluation.validation_dates.equals(sarimax_dates[45:])

### 14.2 GARCH 真实样本外评估

GARCH 的预测量是条件标准差。可观测评估目标定义为绝对去均值收益
`abs(y_t - mean(y_train))`，而不是不可观测的条件波动率，也不是平方残差。

In [ ]:
# GARCH 真实样本外评估
sim = simulate_garch(n=500, p=1, q=1, omega=0.1, alpha=0.2, beta=0.7,
                     seed=42, burn=300)
model = GARCH(sim.data, p=1, q=1)
estimation_end = int(len(sim.data) * 0.7) - 1
evaluation = model.oos(
    estimation_period=(0, estimation_end),
    validation_period=(estimation_end + 1, len(sim.data) - 1),
)

print('=== GARCH 样本外评估指标 ===')
print('评估目标:', evaluation.target)
for name, value in evaluation.metrics.items():
    print(f'  {name:<10s}: {value:.4f}')

fig, ax = plt.subplots(figsize=(8, 3.5))
idx = evaluation.validation_indices
ax.plot(idx, evaluation.actual, color='black', linewidth=0.7,
        alpha=0.45, label='Absolute demeaned return proxy')
ax.plot(idx, evaluation.mean, color='#D55E00', linewidth=1.4,
        label='OOS conditional volatility forecast')
ax.axvline(estimation_end + 1, color='gray', linestyle='--', linewidth=0.8)
ax.legend(fontsize=8)
ax.set_title('GARCH Leakage-Free OOS Volatility Forecast')
plt.show()

### 14.3 VAR 真实样本外评估

多变量结果同时给出合并指标和 `metrics_by_series`，避免只看总体误差掩盖
某个序列的预测失效。

In [ ]:
# VAR 真实样本外评估
r0 = simulate_sarima(n=200, order=(1, 0, 0), ar=[0.7], seed=42, burn=100)
r1 = simulate_sarima(n=200, order=(1, 0, 0), ar=[0.5], seed=99, burn=100)
data_2d = np.column_stack([r0.data, r1.data])

model = VAR(data_2d, lags=2, cols=['y0', 'y1'])
estimation_end = int(len(data_2d) * 0.7) - 1
evaluation = model.oos(
    estimation_period=(0, estimation_end),
    validation_period=(estimation_end + 1, len(data_2d) - 1),
)

print('=== VAR 样本外合并指标 ===')
for name, value in evaluation.metrics.items():
    print(f'  {name:<10s}: {value:.4f}')
print()
print('逐序列 RMSE:')
for col, metrics in zip(['y0', 'y1'], evaluation.metrics_by_series):
    print(f'  {col}: {metrics["rmse"]:.4f}')

fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
idx = evaluation.validation_indices
for i, (ax, col) in enumerate(zip(axes, ['y0', 'y1'])):
    ax.plot(idx, evaluation.actual[:, i], color='black', linewidth=0.8,
            alpha=0.5, label='Actual')
    ax.plot(idx, evaluation.mean[:, i], color='#D55E00', linewidth=1.5,
            label='OOS forecast')
    ax.axvline(estimation_end + 1, color='gray', linestyle='--', linewidth=0.8)
    ax.set_ylabel(col)
    ax.legend(fontsize=7)
axes[0].set_title('VAR Leakage-Free OOS Forecast')
plt.tight_layout()
plt.show()

### 14.4 统一估计期和验证期的多模型 OOS 比较

`evaluate_models_oos()` 对每个模型传入同一组闭区间边界，返回完整指标表、排名、最佳模型及每个模型的 `OOSResult`。
批量比较要求预测目标、期间和实际观测完全一致，并拒绝非有限预测，避免模型通过少算困难观测获得更优排名。

这里的 `evaluate_models_oos()` 比较样本外预测误差；前文的 `compare_models()` 比较已拟合模型的参数估计与显著性，两者用途不同。

In [ ]:
from Ts.TsMetrics import evaluate_models_oos
from Ts.TsModels import SARIMAX
from Ts.TsSims import simulate_sarima

comparison_data = simulate_sarima(
    n=120,
    order=(1, 0, 0),
    ar=[0.65],
    seed=2026,
    burn=100,
).data
comparison_models = {
    'AR(1)': SARIMAX(comparison_data, order=(1, 0, 0), trend='n'),
    'AR(2)': SARIMAX(comparison_data, order=(2, 0, 0), trend='n'),
}

oos_report = evaluate_models_oos(
    comparison_models,
    estimation_period=(0, 89),
    validation_period=(90, 119),
    rank_by='rmse',
)

print(oos_report.table.to_string())
print('排名:', oos_report.ranking)
print('最佳模型:', oos_report.best_model)

assert oos_report.table.columns.tolist() == [
    'mae', 'mse', 'rmse', 'mape', 'smape', 'theil_u1', 'n', 'rank'
]
assert all(result.metrics['n'] == 30 for result in oos_report.evaluations.values())
assert all(
    result.estimation_indices[[0, -1]].tolist() == [0, 89]
    and result.validation_indices[[0, -1]].tolist() == [90, 119]
    for result in oos_report.evaluations.values()
)
assert all(model.result_ is None for model in comparison_models.values())

### 14.5 样本外评估与未来外推分离

有真实值的持出区间用 `model.oos()` 评估；没有真实值的未来区间则先用全样本
拟合，再调用 `result.predict(start=nobs, end=...)`。两类结果不混入同一容器。

In [ ]:
# SARIMAX 未来外推：使用全样本拟合，不计算虚构的评估指标
sim = simulate_sarima(n=200, order=(1, 0, 0), ar=0.7, seed=42, burn=100)
result = SARIMAX(sim.data, order=(1, 0, 0)).fit()

forecast_steps = 10
future = result.predict(
    start=result.nobs,
    end=result.nobs + forecast_steps - 1,
)
future_idx = np.arange(result.nobs, result.nobs + forecast_steps)

print(f'样本终点: t={result.nobs - 1}')
print(f'未来预测: t={future_idx[0]} 到 t={future_idx[-1]}')
print('未来没有实际值，因此 PredictResult 不提供 actual 或 metrics。')

fig, ax = plt.subplots(figsize=(8, 3.5))
observed_idx = np.arange(result.nobs)
ax.plot(observed_idx, sim.data, color='black', linewidth=0.8,
        alpha=0.55, label='Actual')
ax.plot(future_idx, future.mean, color='#009E73', linewidth=1.5,
        marker='o', markersize=4, label='Future forecast')
if future.lower is not None and future.upper is not None:
    ax.fill_between(future_idx, future.lower, future.upper,
                    alpha=0.15, color='#009E73')
ax.axvline(result.nobs - 1, color='gray', linestyle=':', linewidth=0.8,
           label='Sample end')
ax.legend(fontsize=8)
ax.set_title('SARIMAX Future Forecast')
plt.show()

---
## 15. STL — 季节趋势分解

STL 使用 LOESS 将一维序列分解为趋势、季节项和残差。`robust=True` 会降低异常值对分解结果的影响。

In [ ]:
time = np.arange(120, dtype=float)
stl_data = 10 + 0.05 * time + 2 * np.sin(2 * np.pi * time / 12)
stl_data[60] += 8

stl_model = STL(stl_data, period=12, robust=True)
stl_result = stl_model.fit()
print(stl_result.summary())
print(f"\n重构最大误差: {np.max(np.abs(stl_result.observed - stl_result.fitted_values - stl_result.residuals)):.3e}")
print(f"异常点权重: {stl_result.weights[60]:.3f}")
stl_result.plot(title="Monthly STL Decomposition")
plt.show()

---
## 16. Backtesting 与 Backcasting

`backtest()` 在每个历史预测起点只使用当时可见的数据重新拟合，避免未来信息泄漏。`backcast()` 则反转时间、重新拟合并估计样本前数值；它是反向时间统计估计，不是因果历史重建。

In [ ]:
from Ts.TsModels import AutoSARIMAX, SARIMAX

# AutoSARIMAX 在每个预测起点重新选阶，并对齐训练/未来外生变量
evaluation_model = AutoSARIMAX(
    sarimax_y,
    exog=sarimax_controls.loc[sarimax_dates],
    events=sarimax_events,
    p=(0, 0),
    d=(0, 0),
    q=(0, 0),
    P=(0, 0),
    D=(0, 0),
    Q=(0, 0),
)
backtest_result = evaluation_model.backtest(
    initial_window=42,
    horizon=3,
    step=6,
)

# Backcast 示例使用不含外生变量的 SARIMAX 特例，避免猜测样本前 exog
backcast_data = simulate_sarima(
    n=100,
    order=(1, 0, 0),
    ar=[0.7],
    seed=2026,
    burn=100,
).data
backcast_result = SARIMAX(backcast_data, order=(1, 0, 0)).backcast(steps=4)

print("Backtest origins:", backtest_result.origins)
print("Backtest forecast shape:", backtest_result.mean.shape)
print("Backtest RMSE:", f"{backtest_result.metrics['rmse']:.4f}")
print("Backcast indices:", backcast_result.indices)
print("Backcast estimates:", np.round(backcast_result.mean, 4))

assert backtest_result.mean.shape == (3, 3)
assert backcast_result.indices.tolist() == [-4, -3, -2, -1]